# Neutron Star Merger from Wave Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/notebooks/LFM_Neutron_Star_Merger.ipynb)

## What This Notebook Demonstrates

Two dense $\chi$-wells (neutron stars) on a 1D lattice:

1. **Inspiral** via $\chi$-gradient forces
2. **Chirp**: increasing GW frequency as separation shrinks
3. **Merger**: peak amplitude when sources overlap
4. **Ringdown**: post-merger oscillations
5. **QGP transition**: if core density exceeds threshold

Compared to GW170817 phenomenology. Uses GOV-01 + GOV-02 only.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters (reduced for Colab)
chi0, kappa, c = 19.0, 0.016, 1.0
N, L = 400, 200.0
dx = L / N
dt = 0.2 * dx / c
n_steps = 12000
ns_radius = 5.0
ns_amp = 50.0
sep_init = 30.0
ns_mass = 10.0
kappa_dense = 0.05
density_threshold = 3000

x = np.linspace(0, L, N)
center = L / 2
pos1, pos2 = center - sep_init/2, center + sep_init/2
vel1, vel2 = 0.0, 0.0

E = np.zeros(N)
E_prev = np.zeros(N)
monitor_idx = N // 4

separations, gw_signal, times = [], [], []
merger_time = None

for step in range(n_steps):
    t_now = step * dt
    # Sources
    src1 = ns_amp * np.exp(-((x - pos1) / ns_radius)**2)
    src2 = ns_amp * np.exp(-((x - pos2) / ns_radius)**2)
    E_sq = (E + src1 + src2)**2
    # chi field
    kappa_eff = np.where(E_sq > density_threshold, kappa_dense, kappa)
    chi_sq = chi0**2 - kappa_eff * E_sq
    chi = np.sqrt(np.maximum(chi_sq, 0.01))
    # GOV-01 evolution
    lap = np.zeros(N)
    lap[1:-1] = (E[:-2] + E[2:] - 2*E[1:-1]) / dx**2
    E_new = 2*E - E_prev + dt**2 * (c**2 * lap - chi**2 * E)
    E_new[0] = E_new[-1] = 0
    E_prev, E = E.copy(), E_new.copy()
    # Forces on NS
    chi_grad = np.gradient(chi, dx)
    i1, i2 = int(np.clip(pos1/dx, 1, N-2)), int(np.clip(pos2/dx, 1, N-2))
    F1 = -ns_mass * chi_grad[i1]
    F2 = -ns_mass * chi_grad[i2]
    vel1 += F1/ns_mass * dt
    vel2 += F2/ns_mass * dt
    pos1 = np.clip(pos1 + vel1*dt, 5*dx, L - 5*dx)
    pos2 = np.clip(pos2 + vel2*dt, 5*dx, L - 5*dx)
    sep = abs(pos2 - pos1)
    separations.append(sep)
    gw_signal.append(chi[monitor_idx])
    times.append(t_now)
    if merger_time is None and sep < 2 * ns_radius:
        merger_time = t_now

separations = np.array(separations)
gw_signal = np.array(gw_signal)
times = np.array(times)

print('=' * 60)
print('NEUTRON STAR MERGER RESULTS')
print('=' * 60)
print(f'Initial separation: {separations[0]:.1f}')
print(f'Minimum separation: {separations.min():.1f}')
inspiral = separations.min() < separations[0] * 0.5
print(f'Inspiral detected:  {inspiral}')
print(f'Merger time:        {merger_time}')
merged = merger_time is not None
print(f'H0 REJECTED: {merged and inspiral}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax = axes[0]
ax.plot(times, separations, 'b-', lw=1.5)
if merger_time is not None:
    ax.axvline(merger_time, color='red', ls='--', label=f'Merger t={merger_time:.0f}')
ax.set_xlabel('Time')
ax.set_ylabel('Separation')
ax.set_title('NS Inspiral: Separation vs Time')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(times, gw_signal - np.mean(gw_signal), 'r-', lw=0.5)
if merger_time is not None:
    ax.axvline(merger_time, color='blue', ls='--', label='Merger')
ax.set_xlabel('Time')
ax.set_ylabel('chi signal (GW proxy)')
ax.set_title('Gravitational Wave Signal at Monitor Point')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Neutron Star Merger from GOV-01 + GOV-02', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Result

- Two dense $\chi$-wells inspiral under gradient forces
- GW-like signals emerge at the monitor point
- If density exceeds threshold, enhanced $\kappa$ mimics QGP phase transition
- The full GW170817-like sequence emerges from GOV-01 + GOV-02